# Lesson 5.2 — Escalation & Ambiguity Resolution Patterns

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Identify valid escalation triggers: explicit human request, policy gaps, no progress
- Honor explicit human requests immediately without investigating first
- Explain why sentiment and self-reported confidence are poor escalation proxies
- Ask for additional identifiers when tool results return multiple matches

**You will produce:** An escalation decision function with few-shot criteria and a multi-match clarification path.

## Concepts

Appropriate **escalation triggers**: the customer **explicitly requests a human**; a **policy exception/gap** (not just a complex case); or **inability to make meaningful progress**.

Key judgments (Sample Question 3):
- **Honor explicit human requests immediately** - don't investigate first.
- Acknowledge frustration but **offer to resolve** when the issue is within capability; escalate only if the customer reiterates.
- **Sentiment-based escalation and self-reported confidence are unreliable** proxies for actual case complexity - the agent is already over-confident on hard cases.
- Escalate when **policy is ambiguous/silent** (e.g. competitor price-matching when policy only addresses own-site adjustments).
- On **multiple customer matches**, ask for **additional identifiers** - never pick heuristically.

The right first fix for miscalibrated escalation is **explicit criteria + few-shot examples**, not a classifier or sentiment model.

> **Reference:** [Anthropic - Agent SDK / guardrails](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Encode escalation triggers

Three concrete triggers; sentiment and confidence are explicitly excluded.

In [ ]:
def should_escalate(ctx):
    if ctx.get("explicit_human_request"):
        return (True, "customer explicitly asked for a human")
    if ctx.get("policy_gap"):
        return (True, "policy is ambiguous/silent on this request")
    if ctx.get("no_progress"):
        return (True, "unable to make meaningful progress")
    return (False, "resolve autonomously")

print(should_escalate({"explicit_human_request": True}))
print(should_escalate({"policy_gap": True}))
print(should_escalate({"sentiment": "angry"}))  # NOT a trigger

## Code-Along: Step 2 — Few-shot escalate-vs-resolve examples

Examples pin the boundary the way the exam recommends.

In [ ]:
ESC_SHOTS = [
    {"case": "standard damaged item with photo", "action": "resolve"},
    {"case": "customer: 'let me talk to a person'", "action": "escalate now"},
    {"case": "competitor price-match, policy silent", "action": "escalate (policy gap)"},
]
for s in ESC_SHOTS:
    print(f"{s['case']} -> {s['action']}")

## Code-Along: Step 3 — Clarify on multiple matches

Two accounts match -> request another identifier rather than guessing.

In [ ]:
def resolve_customer(matches):
    if len(matches) > 1:
        return {"action": "ask_identifier",
                "message": "I found multiple accounts. Can you share your order # or ZIP?"}
    if not matches:
        return {"action": "ask_identifier", "message": "No match - please verify your email."}
    return {"action": "proceed", "customer": matches[0]}

print(resolve_customer([{"id": "A"}, {"id": "B"}]))
print(resolve_customer([{"id": "A"}]))

## Exercise

Extend `should_escalate` to the 'acknowledge then offer' path: when the customer is frustrated but the issue is within capability, return 'offer_resolution' and only escalate if `reiterated` is True. Assert that a frustrated-but-solvable case is NOT escalated on the first turn, and that an explicit human request always escalates immediately.

In [ ]:
# TODO: add 'offer_resolution' and reiteration handling to should_escalate
# TODO: assert frustrated+solvable -> not escalate on first turn
# TODO: assert explicit_human_request -> escalate immediately

## Key Takeaways

- Escalate on explicit human request, policy gaps, or inability to progress.
- Honor explicit human requests immediately; don't investigate first.
- Sentiment and self-reported confidence are poor proxies for case complexity.
- On multiple matches, ask for more identifiers instead of guessing.

---

**Next:** [Lesson 5.3 — Error Propagation Across Multi-Agent Systems](./lesson-5.3-error-propagation.ipynb)